<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Space X  Falcon 9 First Stage Landing Prediction**


## Web scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia


Estimated time needed: **40** minutes


In this lab, you will be performing web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/Falcon9_rocket_family.svg)


Falcon 9 first stage will land successfully


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/crash.gif)


More specifically, the launch records are stored in a HTML table shown below:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/falcon9-launches-wiki.png)


  ## Objectives
Web scrap Falcon 9 launch records with `BeautifulSoup`: 
- Extract a Falcon 9 launch records HTML table from Wikipedia
- Parse the table and convert it into a Pandas data frame


First let's import required packages for this lab


In [1]:
!pip3 install beautifulsoup4
!pip3 install requests

In [2]:
import sys

import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

and we will provide some helper functions for you to process web scraped HTML table


In [3]:
def date_time(table_cells):
    """
    This function returns the data and time from the HTML  table cell
    Input: the  element of a table data cell extracts extra row
    """
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """
    This function returns the booster version from the HTML  table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=''.join([booster_version for i,booster_version in enumerate( table_cells.strings) if i%2==0][0:-1])
    return out

def landing_status(table_cells):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=[i for i in table_cells.strings][0]
    return out


def get_mass(table_cells):
    mass=unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass=mass[0:mass.find("kg")+2]
    else:
        new_mass=0
    return new_mass


def extract_column_from_header(row):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    if (row.br):
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
        
    colunm_name = ' '.join(row.contents)
    
    # Filter the digit and empty names
    if not(colunm_name.strip().isdigit()):
        colunm_name = colunm_name.strip()
        return colunm_name    


To keep the lab tasks consistent, you will be asked to scrape the data from a snapshot of the  `List of Falcon 9 and Falcon Heavy launches` Wikipage updated on
`9th June 2021`


In [4]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/91.0.4472.124 Safari/537.36"
}

Next, request the HTML page from the above URL and get a `response` object


### TASK 1: Request the Falcon9 Launch Wiki page from its URL


First, let's perform an HTTP GET method to request the Falcon9 Launch HTML page, as an HTTP response.


In [5]:
# use requests.get() method with the provided static_url and headers
# assign the response to a object
# POSS.ANS.:
response = requests.get(static_url, headers=headers)
#response # This does not output anything
#print(response.text)  # This works

Create a `BeautifulSoup` object from the HTML `response`


In [6]:
from bs4 import BeautifulSoup
# Use BeautifulSoup() to create a BeautifulSoup object from a response text content
# POSS.ANS.:
#soup = BeautifulSoup(response, 'html5lib')
#soup = BeautifulSoup(response, 'html.parser')
soup = BeautifulSoup(response.text, 'html.parser')

Print the page title to verify if the `BeautifulSoup` object was created properly 


In [7]:
# Use soup.title attribute
# POSS.ANS.:
#soup
print(soup.title)

<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>


### TASK 2: Extract all column/variable names from the HTML table header


Next, we want to collect all relevant column names from the HTML table header


Let's try to find all tables on the wiki page first. If you need to refresh your memory about `BeautifulSoup`, please check the external reference link towards the end of this lab


In [8]:
# Use the find_all function in the BeautifulSoup object, with element type `table`
# Assign the result to a list called `html_tables`
# POSS.ANS.:
html_tables = soup.find_all('table')


Starting from the third table is our target table contains the actual launch records.


In [9]:
# Let's print the third table and check its content
first_launch_table = html_tables[2]
print(first_launch_table)

<table class="wikitable plainrowheaders collapsible" style="width: 100%;">
<tbody><tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">Version,<br/>Booster</a> <sup class="reference" id="cite_ref-booster_11-0"><a href="#cite_note-booster-11"><span class="cite-bracket">[</span>b<span class="cite-bracket">]</span></a></sup>
</th>
<th scope="col">Launch site
</th>
<th scope="col">Payload<sup class="reference" id="cite_ref-Dragon_12-0"><a href="#cite_note-Dragon-12"><span class="cite-bracket">[</span>c<span class="cite-bracket">]</span></a></sup>
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launch<br/>outcome
</th>
<th scope="col"><a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 

You should able to see the columns names embedded in the table header elements `<th>` as follows:


```
<tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">Version,<br/>Booster</a> <sup class="reference" id="cite_ref-booster_11-0"><a href="#cite_note-booster-11">[b]</a></sup>
</th>
<th scope="col">Launch site
</th>
<th scope="col">Payload<sup class="reference" id="cite_ref-Dragon_12-0"><a href="#cite_note-Dragon-12">[c]</a></sup>
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launch<br/>outcome
</th>
<th scope="col"><a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 9 first-stage landing tests">Booster<br/>landing</a>
</th></tr>
```


Next, we just need to iterate through the `<th>` elements and apply the provided `extract_column_from_header()` to extract column name one by one


In [10]:
#Contrast the following cells:

In [11]:
col_names = []
for row in first_launch_table.find_all('th'):
    text_content = extract_column_from_header(row)  # Uses given function
    col_names.append(text_content)
print("col_names: ", col_names)

col_names:  ['Flight No.', 'Date and time ( )', '', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', '', None, None, None, None, None, None, None]


In [12]:
# THEE POSS.ANS.:
col_names = []
for row in first_launch_table.find_all('th'):
    text_content = extract_column_from_header(row)  # Uses given function
    if text_content is not None and len(text_content) > 0:
        col_names.append(text_content)
print("col_names: ", col_names)

col_names:  ['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


In [13]:
column_names = []
headers = first_launch_table.find_all('th')
print("headers: ", headers)
column_names = [header.text.strip() for header in headers]
print("column_names: ", column_names)


headers:  [<th scope="col">Flight No.
</th>, <th scope="col">Date andtime ()
</th>, <th scope="col"> 
</th>, <th scope="col">Launch site
</th>, <th scope="col">Payload
</th>, <th scope="col">Payload mass
</th>, <th scope="col">Orbit
</th>, <th scope="col">Customer
</th>, <th scope="col">Launchoutcome
</th>, <th scope="col">
</th>, <th rowspan="2" scope="row" style="text-align:center;">1
</th>, <th rowspan="2" scope="row" style="text-align:center;">2
</th>, <th rowspan="2" scope="row" style="text-align:center;">3
</th>, <th rowspan="3" scope="row" style="text-align:center;">4
</th>, <th rowspan="2" scope="row" style="text-align:center;">5
</th>, <th rowspan="2" scope="row" style="text-align:center;">6
</th>, <th rowspan="2" scope="row" style="text-align:center;">7
</th>]
column_names:  ['Flight No.', 'Date andtime ()', '', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launchoutcome', '', '1', '2', '3', '4', '5', '6', '7']


In [14]:
# Print the title
print(soup.title.string)

# Print the first heading (h1)
print(soup.h1.string)

# Print all paragraph texts
for p in soup.find_all('p'):
    print(p.text)


List of Falcon 9 and Falcon Heavy launches - Wikipedia
List of Falcon 9 and Falcon Heavy launches


Since June 2010, rockets from the Falcon 9 family have been launched 621 times, with 618 full mission successes, one partial failure and one total loss of spacecraft. In addition, one rocket and its payload were destroyed on the launch pad during the fueling process before a static fire test.

Designed and operated by private manufacturer SpaceX, the Falcon 9 rocket family includes the retired versions Falcon 9 v1.0, v1.1, and v1.2 "Full Thrust" Block 1 to 4, along with the currently active Block 5 evolution. Falcon Heavy is a heavy-lift derivative of Falcon 9, combining a strengthened central core with two Falcon 9 first stages as side boosters.[1]

The Falcon design features reusable first-stage boosters, which land either on a ground pad near the launch site or on a drone ship at sea.[2] In December 2015, Falcon 9 became the first rocket to land propulsively after delivering a payload

In [15]:
# Just DON'T EXECUTE THE CELL BELOW, instead of commenting out each line of the cell

In [16]:
column_names = []

# Apply find_all() function with `th` element on first_launch_table
# Iterate each th element and apply the provided extract_column_from_header() to get a column name
# Append the Non-empty column name (`if name is not None and len(name) > 0`) into a list called column_names
# POSS.ANS.:
headers = first_launch_table.find_all('th')
#column_names = [header.extract_column_from_header() for header in headers]



#if table:
#    # 1. Extract the headers (optional, for better structure)
#    headers = [header.get_text(strip=True) for header in table.find_all('th')]
#    print("Headers:", headers)

#    # 2. Extract the data rows
#    rows = []
#    # Find all <tr> tags within the table's body (<tbody>)
#    for row in table.find_all('tr'):
#        # Find all <td> tags within each row
#        cols = [col.get_text(strip=True) for col in row.find_all('td')]
#        
#        # Only add rows that have data cells (skips the header row if not in <thead>)
#        if cols:
#            rows.append(cols)

#   # Print the extracted data
#    print("Extracted Data:", rows)
#else:
#    print("Table not found.")
    
    

## 1. Extract the headers (optional, for better structure)
#headers = [header.get_text(strip=True) for header in html_tables[3].find_all('th')]
#print("Headers:", headers)

# 1. Initialize an empty list to store the results.
headers = []

# 2. Loop through every element returned by table.find_all('th').
#    table.find_all('th') finds all the <th> tags inside the table object.
for header in html_tables[3].find_all('th'):
    
    # 3. For each <th> element, get its text content and remove leading/trailing whitespace.
    text_content = header.get_text(strip=True)
    
    # 4. Append the cleaned text to the new list.
    headers.append(text_content)



# 2. Extract the data rows
    rows = []
    # Find all <tr> tags within the table's body (<tbody>)
    for row in html_tables[2].find_all('tr'):
        # Find all <td> tags within each row
        #column_names = [extract_column_from_header(row) for column_names in row.find_all('td'
    
        for column_names in row.find_all('tr'):
            text_content = extract_column_from_header(row)
            rows.append(text_content)
    
    
    
    
    # Only add rows that have data cells (skips the header row if not in <thead>)
    if row != None and len(row) > 0:
        rows.append(column_names)
        #column_names = append(column_names)
            
# Print the extracted data
print("Extracted Data:", rows)

    
    
# Consider first_launch_table (html_tables[2]):
#if table:
#  # Find all 'th' tags
#  column_names = [extract_column_from_header(row) for html_tables[3].find_all('th')]
#
#  rows = []


#  if name != None and len(name) > 0:
#    column_names = append(column_names)

    
  
    
  





Extracted Data: [[]]


Check the extracted column names


In [17]:
#print(column_names)

## TASK 3: Create a data frame by parsing the launch HTML tables


We will create an empty dictionary with keys from the extracted column names in the previous task. Later, this dictionary will be converted into a Pandas dataframe


In [18]:
launch_dict= dict.fromkeys(column_names)

# Remove an irrelvant column
#del launch_dict['Date and time ( )']
#del launch_dict['Date andtime ( )']
#del launch_dict['Date andtime ()']

# Let's initial the launch_dict with each value to be an empty list
launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []
# Added some new columns
launch_dict['Version Booster']=[]
launch_dict['Booster landing']=[]
launch_dict['Date']=[]
launch_dict['Time']=[]

Next, we just need to fill up the `launch_dict` with launch records extracted from table rows.


Usually, HTML tables in Wiki pages are likely to contain unexpected annotations and other types of noises, such as reference links `B0004.1[8]`, missing values `N/A [e]`, inconsistent formatting, etc.


To simplify the parsing process, we have provided an incomplete code snippet below to help you to fill up the `launch_dict`. Please complete the following code snippet with TODOs or you can choose to write your own logic to parse all launch tables:


In [27]:
extracted_row = 0
#Extract each table 
for table_number,table in enumerate(soup.find_all('table',"wikitable plainrowheaders collapsible")):
   # get table row 
    for rows in table.find_all("tr"):
        #check to see if first table heading is as number corresponding to launch a number 
        if rows.th:
            if rows.th.string:
                flight_number=rows.th.string.strip()
                flag=flight_number.isdigit()
        else:
            flag=False
        #get table element 
        row=rows.find_all('td')
        #if it is number save cells in a dictonary 
        if flag:
            extracted_row += 1
            # Flight Number value
            # TODO: Append the flight_number into launch_dict with key `Flight No.`
            #print(flight_number)
            #flight_number.append(launch_dict['Flight No.'])
            launch_dict['Flight No.'].append(flight_number)
            print(flight_number)
            
            datatimelist=date_time(row[0])
            
            # Date value
            # TODO: Append the date into launch_dict with key `Date`
            date = datatimelist[0].strip(',')
            #print(date)
            launch_dict['Date'].append(date)
            print(date)

            
            # Time value
            # TODO: Append the time into launch_dict with key `Time`
            time = datatimelist[1]
            #print(time)
            launch_dict['Time'].append(time)
            print(time)

                
            # Booster version
            # TODO: Append the bv into launch_dict with key `Version Booster`
            #START OVER:
            #print("ROW[1]: ", row[1]) #DIAGNOSTIC
            bv = row[1].a.string
            launch_dict['Version Booster'].append(bv)
            print(bv)                                  #seems better
            
            
#            print("ROW[1]: ", row[1]) #DIAGNOSTIC
#            if not(bv):
#                bv=row[1].a.string
            #print(bv)
            
            #launch_dict['Version Booster'].append(booster_version)
            #print(booster_version)

#            launch_dict['Version Booster'].append(bv)
            #print(bv)
#            print(booster_version)
            
            
            # Launch Site
            # TODO: Append the bv into launch_dict with key `Launch Site`
            #print("ROW[2]: ", row[2]) #DIAGNOSTIC
#            launch_site = row[2].a.string
#            #print(launch_site)
#            launch_dict['Launch site'].append(launch_site)
#            print(launch_site)

            bv = row[2].a.string
            launch_dict['Launch site'].append(bv)
            print(bv)
            
            
            
            
            
            
            # Payload
            # TODO: Append the payload into launch_dict with key `Payload`
            payload = row[3].a.string
            #print(payload)
            launch_dict['Payload'].append(payload)
            print(payload)


            
            # Payload Mass
            # TODO: Append the payload_mass into launch_dict with key `Payload mass`
            payload_mass = get_mass(row[4])
            #print(payload)
            launch_dict['Payload mass'].append(payload_mass)
            print(payload_mass)

            
            
            # Orbit
            # TODO: Append the orbit into launch_dict with key `Orbit`
            orbit = row[5].a.string
            #print(orbit)
            launch_dict['Orbit'].append(orbit)
            print(orbit)

            
            
            # Customer
            # TODO: Append the customer into launch_dict with key `Customer`
            
            # BELOW is in response to a 'Customer' cell of None, which shows up as an error if not addressed:
            # The fairly common BeautifulSoup() error:
            # AttributeError: 'NoneType' object has no attribute 'string'
            #print("ROW[6]: ", row[6]) #DIAGNOSTIC
            customer = row[6].a.string
            launch_dict['Customer'].append(customer)
            print(customer)
            
            
            
            
            
            #if a is None or len(a) == 0:
                #a = "NONE_CUST"
                #print("CUSTOMER is None")

                
            #if a is None or len(a) == 0 or a is NaN:
            #if a or a.string is None:
            #    customer = "NONE_CUST"
            #
            #   
            #else:
            #    customer = row[6].a.string
            customer = row[6].a.string    
            #print(customer)
            
            
            
            # Avoid the error by checking if the variable is None
            #if p_tag and p_tag.string is not None:
            #    print(f"The string for p_tag is: '{p_tag.string}'")
            #else:
            #    print("Could not get a single string for p_tag.")

            #if h2_tag:
            #    print(f"The string for h2_tag is: '{h2_tag.string}'")
            #else:
            #    print("No <h2> tag was found, so the tag is None.")


            

            
            launch_dict['Customer'].append(customer)
            print(customer)

            
            
            # Launch outcome
            # TODO: Append the launch_outcome into launch_dict with key `Launch outcome`
            launch_outcome = list(row[7].strings)[0]
            #print(launch_outcome)
            launch_dict['Launch outcome'].append(launch_outcome)
            print(launch_outcome)

            
            
            # Booster landing
            # TODO: Append the launch_outcome into launch_dict with key `Booster landing`
            booster_landing = landing_status(row[8])
            #print(booster_landing)
            launch_dict['Booster landing'].append(booster_landing)
            print(booster_landing)

            
            

1
4 June 2010
18:45
F9 v1.0
CCAFS
Dragon Spacecraft Qualification Unit
0
LEO
SpaceX
SpaceX
Success

Failure
2
8 December 2010
15:43
F9 v1.0
CCAFS
Dragon
0
LEO
NASA
NASA
Success
Failure
3
22 May 2012
07:44
F9 v1.0
CCAFS
Dragon
525 kg
LEO
NASA
NASA
Success
No attempt

4
8 October 2012
00:35
F9 v1.0
CCAFS
SpaceX CRS-1
4,700 kg
LEO
NASA
NASA
Success

No attempt
5
1 March 2013
15:10
F9 v1.0
CCAFS
SpaceX CRS-2
4,877 kg
LEO
NASA
NASA
Success

No attempt

6
29 September 2013
16:00
F9 v1.1
VAFB
CASSIOPE
500 kg
Polar orbit
MDA
MDA
Success
Uncontrolled
7
3 December 2013
22:41
F9 v1.1
CCAFS
SES-8
3,170 kg
GTO
SES
SES
Success
No attempt
8
6 January 2014
22:06
F9 v1.1
CCAFS
Thaicom 6
3,325 kg
GTO
Thaicom
Thaicom
Success
No attempt
9
18 April 2014
19:25
F9 v1.1
Cape Canaveral
SpaceX CRS-3
2,296 kg
LEO
NASA
NASA
Success

Controlled
10
14 July 2014
15:15
F9 v1.1
Cape Canaveral
Orbcomm-OG2
1,316 kg
LEO
Orbcomm
Orbcomm
Success
Controlled
11
5 August 2014
08:00
F9 v1.1
Cape Canaveral
AsiaSat 8
4,535 kg
GT

AttributeError: 'NoneType' object has no attribute 'string'

After you have fill in the parsed launch record values into `launch_dict`, you can create a dataframe from it.


In [28]:
df= pd.DataFrame({ key:pd.Series(value) for key, value in launch_dict.items() })

We can now export it to a <b>CSV</b> for the next section, but to make the answers consistent and in case you have difficulties finishing this lab. 

Following labs will be using a provided dataset to make each lab independent. 


<code>df.to_csv('spacex_web_scraped.csv', index=False)</code>


In [29]:
df.to_csv('spacex_web_scraped.csv', index=False)

## Authors


<a href="https://www.linkedin.com/in/yan-luo-96288783/">Yan Luo</a>


<a href="https://www.linkedin.com/in/nayefaboutayoun/">Nayef Abou Tayoun</a>


<!--
## Change Log
-->


<!--
| Date (YYYY-MM-DD) | Version | Changed By | Change Description      |
| ----------------- | ------- | ---------- | ----------------------- |
| 2021-06-09        | 1.0     | Yan Luo    | Tasks updates           |
| 2020-11-10        | 1.0     | Nayef      | Created the initial version |
-->


Copyright © 2021 IBM Corporation. All rights reserved.
